# Exploring Hamilton DAGs with iacs Data

Hamilton is a framework for defining dataflows (DAGs) using plain Python functions:
- **Function names** become node names
- **Parameter names** declare dependencies on other nodes
- The DAG is constructed lazily — nothing executes until you call `driver.execute()`

Here we define a small Hamilton dataflow over iacs registry tables and visualize the DAG.

In [ ]:
import ibis
from hamilton import driver, base
from iacs.dataflows import load_manifest, validate_registry, base_etl
from iacs.dataflows.audit import requirement_coverage

ibis.options.interactive = True

In [ ]:
input_dir = "../examples/example"

In [ ]:
target_module = base_etl

In [ ]:
import importlib
importlib.reload(target_module)

In [ ]:
config = {}
dr = driver.Driver(config, target_module)

In [ ]:
dr.display_all_functions()

In [ ]:

dr.visualize_execution([base_etl.loaded_registry], inputs={"input_dir": input_dir})

In [ ]:
dr.visualize_path_between("loaded_registry.spine", "validated_registry.updated_components", strict_path_visualization=False)

## Chaining and Re-using DAGs

Hamilton supports two complementary patterns for composing pipelines:

1. **Multi-module chaining** — pass several modules to a single `Driver`. Hamilton merges the DAGs, so outputs from one module automatically flow as inputs to the next. A shared node name (e.g. `registry`) is computed once and reused wherever it is referenced.

2. **Sequential re-use** — run one driver to extract an intermediate result, then pass it as an external input to a second driver. This is useful when you want to run independent downstream pipelines without re-running the upstream work.

### Approach 1: Multi-module chaining

Pass `load_manifest` and `validate_registry` to the same `Driver`. Hamilton merges their node graphs — `load_manifest.registry` satisfies `validate_registry`'s `registry` parameter with no glue code needed.

In [ ]:
dr_chain = driver.Driver(
    {"input_dir": ["../examples/example"]},
    load_manifest,
    validate_registry,
    adapter=base.DictResult(),
)
dr_chain  # visualize the merged DAG

In [ ]:
# Execute the full chain with a single call — load_manifest runs first, its registry
# flows into validate_registry automatically.
chain_result = dr_chain.execute(["validated_registry"])
validated_reg = chain_result["validated_registry"]
print("Component types in validated registry:", validated_reg.component_types)

### Approach 2: Sequential re-use

Run `load_manifest` once and extract `registry`. Then pass it as an external input to *multiple* independent drivers. Each downstream driver gets the same result without re-running the upstream computation.

In [ ]:
# Step 1: run load_manifest once and capture the registry.
registry = driver.Driver(
    {"input_dir": ["../examples/example"]},
    load_manifest,
    adapter=base.DictResult(),
).execute(["registry"])["registry"]

# Step 2: re-use the same registry across independent downstream drivers.
dr_validate = driver.Driver({"registry": registry}, validate_registry, adapter=base.DictResult())
dr_audit    = driver.Driver({"registry": registry}, requirement_coverage, adapter=base.DictResult())

# Both drivers share the registry without re-running load_manifest.
print("validate_registry nodes :", [v.name for v in dr_validate.list_available_variables() if not v.is_external_input])
print("requirement_coverage nodes:", [v.name for v in dr_audit.list_available_variables()    if not v.is_external_input])

### Approach 3: Nodes that run inner drivers

A Hamilton node is just a Python function, so it can create and execute its own inner `Driver`. From the outer DAG's perspective the node is a black box — the sub-pipeline inside is an implementation detail. This is useful for:

- **Encapsulating** a multi-step pipeline as a single composable unit
- **Mixing abstraction levels** — e.g. a high-level "load and validate" step that hides its internals from the outer flow
- **Testing** each stage in isolation without the outer harness

`hamilton.ad_hoc_utils.create_temporary_module` turns a list of plain functions into a Hamilton module without needing a dedicated `.py` file.

In [ ]:
from hamilton.ad_hoc_utils import create_temporary_module
from iacs.registry import Registry

def load_and_validate(input_dir: list[str]) -> Registry:
    """Run the full load + validate pipeline as a single DAG node.

    The inner driver is an implementation detail — the outer DAG just
    sees a function that maps input_dir -> Registry.
    """
    return driver.Driver(
        {"input_dir": input_dir},
        load_manifest,
        validate_registry,
        adapter=base.DictResult(),
    ).execute(["validated_registry"])["validated_registry"]

# Build an outer driver whose only node is load_and_validate.
outer_dr = driver.Driver(
    {"input_dir": ["../examples/example"]},
    create_temporary_module(load_and_validate),
    adapter=base.DictResult(),
)
outer_dr

In [ ]:
result = outer_dr.execute(["load_and_validate"])
validated_reg = result["load_and_validate"]
print("Component types:", validated_reg.component_types)